# CodeGuard AI · Experiment 06: Fine-tuning CodeBERT

**Question:** does a pretrained code model beat TF-IDF on the same data and split as model v2?

**Why CodeBERT?** TF-IDF only counts words. CodeBERT (Feng et al., 2020) was pretrained on millions of functions in six languages including Python, and reads tokens *in order*, so in principle it can notice a missing check or a changed argument.

**Set-up (kept identical to Experiment 05b where possible)**
- Data: the v2 dataset (true before/after pairs, changed code, no test code).
- Split: the same repository-grouped 80/20 split with seed 42. From the training repositories we hold out 15% as a **validation** set to choose the epoch, so the test set is used once.
- Model: `microsoft/codebert-base` + a classification head; inputs truncated to 256 tokens; 3 epochs; AdamW, learning rate 2e-5; batch 16.
- Hardware: CPU only (no GPU available), which is why sequences are capped at 256 tokens.

> Same caveat as 05b: the held-out repositories have been seen in earlier experiments; report this separately.

In [1]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, roc_curve, confusion_matrix)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED, MAX_LEN, EPOCHS, BATCH, LR = 42, 256, 3, 16, 2e-5
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads()))
print("torch", torch.__version__, "| threads", torch.get_num_threads(), "| CUDA", torch.cuda.is_available())

torch 2.14.0+cpu | threads 16 | CUDA False


## Step 1: Rebuild the v2 dataset and split (same code as 05b)

In [2]:
raw = pd.read_csv("../datasets/processed/python_functions.csv")
KEY = ["commit_hash", "file_path", "method_name"]
sizes = raw.groupby(KEY)["target"].agg(n="size", vulnerable="sum")
pairs = raw.set_index(KEY).loc[sizes[(sizes["n"] == 2) & (sizes["vulnerable"] == 1)].index].reset_index()
pairs["code_key"] = pairs["func"].str.replace(r"\s+", "", regex=True)
changed = pairs.groupby(KEY)["code_key"].nunique() == 2
pairs = pairs.set_index(KEY).loc[changed[changed].index].reset_index()
TEST_PATH = re.compile(r"(^|/)(tests?|testing|unit_tests)/|(^|/)test_[^/]*\.py$|_tests?\.py$|(^|/)conftest\.py$")
TEST_NAME = re.compile(r"^(test|_test|setUp|tearDown|setUpClass|tearDownClass)")
is_test = pairs["file_path"].str.contains(TEST_PATH) | pairs["method_name"].str.contains(TEST_NAME)
pairs = pairs[~pairs.groupby(KEY)["file_path"].transform(lambda s: is_test.loc[s.index].any())]
conflict = pairs.groupby("code_key")["target"].nunique()
pairs = pairs[~pairs["code_key"].isin(conflict[conflict > 1].index)]
pairs = pairs[pairs.groupby(KEY)["target"].transform("size") == 2].copy()
pairs["pair_id"] = pairs.groupby(KEY).ngroup()
df = pairs.reset_index(drop=True)

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(df, df["target"], df["repo_url"]))
train_all, test = df.iloc[train_idx], df.iloc[test_idx]
fit_idx, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(train_all, train_all["target"], train_all["repo_url"]))
fit, val = train_all.iloc[fit_idx], train_all.iloc[val_idx]
print(f"v2 dataset {len(df)} | fit {len(fit)} | validation {len(val)} | test {len(test)}")
assert len(test) == 726, "should match experiment 05b's test set" 

C:\Users\Dell\AppData\Local\Temp\ipykernel_24424\192377804.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_test = pairs["file_path"].str.contains(TEST_PATH) | pairs["method_name"].str.contains(TEST_NAME)


v2 dataset 2920 | fit 1700 | validation 494 | test 726


## Step 2: Tokenise (CodeBERT's own tokenizer, truncated to 256 tokens)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
lengths = [len(tokenizer(t, truncation=False)["input_ids"]) for t in df["func"]]
print(f"Median tokens per method: {int(np.median(lengths))}; truncated at {MAX_LEN}: {np.mean(np.array(lengths) > MAX_LEN):.0%} of methods")

def encode(frame):
    enc = tokenizer(list(frame["func"]), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"], torch.tensor(frame["target"].values)

fit_t, val_t, test_t = encode(fit), encode(val), encode(test)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (625 > 512). Running this sequence through the model will result in indexing errors


Median tokens per method: 441; truncated at 256: 67% of methods


## Step 3: Fine-tune, choosing the epoch on the validation set

In [4]:
model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

def scores(tensors, batch=32):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(tensors[0]), batch):
            logits = model(input_ids=tensors[0][i:i + batch], attention_mask=tensors[1][i:i + batch]).logits
            out.append(torch.softmax(logits, dim=-1)[:, 1])
    return torch.cat(out).numpy()

history, best_auc, best_state = [], -1, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(fit_t[0]))
    started, losses = time.time(), []
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        loss = model(input_ids=fit_t[0][idx], attention_mask=fit_t[1][idx], labels=fit_t[2][idx]).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
    val_auc = roc_auc_score(val_t[2].numpy(), scores(val_t))
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_roc_auc": float(val_auc),
                    "minutes": (time.time() - started) / 60})
    print(history[-1])
    if val_auc > best_auc:
        best_auc, best_state = val_auc, {k: v.clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
pd.DataFrame(history)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'epoch': 1, 'train_loss': 0.707146390576229, 'val_roc_auc': 0.494549984428527, 'minutes': 17.310296722253163}


{'epoch': 2, 'train_loss': 0.7004671486738686, 'val_roc_auc': 0.51494041862676, 'minutes': 17.345395195484162}


{'epoch': 3, 'train_loss': 0.6969126072999473, 'val_roc_auc': 0.4983608975724893, 'minutes': 17.437596094608306}


,epoch,train_loss,val_roc_auc,minutes
0,1,0.707146,0.494550,17.310297
1,2,0.700467,0.514940,17.345395
2,3,0.696913,0.498361,17.437596


## Step 4: Threshold with the same 10% false-positive cap (on validation data), then test once

In [5]:
val_scores = scores(val_t)
fpr, tpr, thr = roc_curve(val_t[2].numpy(), val_scores)
ok = np.where(fpr <= 0.10)[0]
threshold = float(thr[ok[np.argmax(tpr[ok])]])

test_scores = scores(test_t)
y_test = test_t[2].numpy()
pred = (test_scores >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
paired = test.assign(score=test_scores).pivot_table(index="pair_id", columns="target", values="score")
codebert = {
    "accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, test_scores),
    "pr_auc": average_precision_score(y_test, test_scores), "false_positive_rate": fp / (fp + tn),
    "pairwise_accuracy": float((paired[1] > paired[0]).mean()), "threshold": threshold,
}
codebert = {k: round(float(v), 3) for k, v in codebert.items()}
tfidf = json.loads(Path("../artifacts/metrics/python_results_v2.json").read_text())["test_metrics"]
comparison = pd.DataFrame({"TF-IDF + LR (05b)": {k: tfidf.get(k) for k in codebert}, "CodeBERT (06)": codebert})
comparison

,TF-IDF + LR (05b),CodeBERT (06)
accuracy,0.511,0.512
precision,0.561,0.569
recall,0.102,0.102
f1,0.172,0.173
roc_auc,0.544,0.521
pr_auc,0.537,0.536
false_positive_rate,0.080,0.077
pairwise_accuracy,0.587,0.424
threshold,NaN,0.494


In [6]:
Path("../artifacts/metrics/codebert_results.json").write_text(json.dumps({
    "model": "microsoft/codebert-base fine-tuned", "max_len": MAX_LEN, "epochs_run": EPOCHS,
    "best_epoch": int(max(history, key=lambda h: h["val_roc_auc"])["epoch"]), "history": history,
    "train_size": len(fit), "validation_size": len(val), "test_size": len(test),
    "test_metrics": codebert, "hardware": "CPU only"}, indent=2))
print("Saved artifacts/metrics/codebert_results.json (the fine-tuned weights are not kept: ~500 MB)")

Saved artifacts/metrics/codebert_results.json (the fine-tuned weights are not kept: ~500 MB)


## Conclusion

| Test, 726 methods (363 pairs) | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC | FPR | Pairwise |
|---|---|---|---|---|---|---|---|---|
| TF-IDF + LR (05b) | 0.511 | 0.561 | 0.102 | 0.172 | **0.544** | 0.537 | 0.080 | **0.587** |
| CodeBERT (06) | 0.512 | 0.569 | 0.102 | 0.173 | 0.521 | 0.536 | 0.077 | 0.424 |

1. **CodeBERT did not learn the task in our set-up.** The training loss stayed at about 0.70, which is the loss of random guessing (ln 2 ≈ 0.693). Validation ROC-AUC was 0.49–0.51 after every epoch, and test ROC-AUC is 0.52. Its pairwise accuracy (0.42) is *below* chance. With the 10% false-positive cap, its precision/recall numbers look like TF-IDF's only because both flag very few methods.
2. **Likely reasons:**
   - only 1,700 training methods, with noisy labels;
   - each vulnerable/fixed pair differs by a median of 5 lines, so the signal is tiny;
   - 67% of methods were cut at 256 tokens (the median is 441), so the changed lines were often removed;
   - CPU-only training limited us to 3 epochs (about 17 minutes each).

   Published work reports similar difficulty on realistic paired data. For example, the PrimeVul benchmark (Ding et al., 2024) found that code language models drop to near-chance pairwise performance.
3. **Decision:** the scanner keeps the TF-IDF model v2. It is better here, about 1,000 times faster, and explainable.
4. **What could work better (future work):** a GPU; more data (all CVEfixes languages, or Python data from other sources); longer inputs (512 tokens or the changed region only); or a model trained to compare the two versions directly.

The fine-tuned weights (~500 MB) are not kept; the notebook rebuilds them.